# Data Ingestion using FastF1


In [ ]:
# Install fastf1 if not already installed
!pip install fastf1

# --------------------------------------------------------------
#  FASTF1 – 2018-2025 → ONE BIG CSV PER DATA TYPE
# --------------------------------------------------------------


In [ ]:

# --------------------------------------------------------------
#  OBJECT-ORIENTED FASTF1 DATA COLLECTION: 2018–2025 → CSVs
# --------------------------------------------------------------

import fastf1 as ff1
from pathlib import Path
import logging
import time
from typing import Any, Dict, List, Optional, Set
import pandas as pd
from fastf1.core import Session

# --- LOGGING ---

logging.basicConfig(

    level=logging.INFO,

    format="%(asctime)s | %(levelname)8s | %(message)s",

    datefmt="%H:%M:%S",

    force=True,

)

logger = logging.getLogger("fastf1-export")

logging.getLogger("fastf1").setLevel(logging.INFO)

# --- PATHS ---
# Get project root (works whether running from notebooks/ or F1/ folder)
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CACHE_DIR = PROJECT_ROOT / "notebooks" / "f1_cache"
SAVE_ROOT = PROJECT_ROOT / "data" / "raw" / "fastf1_2018plus"

SAVE_ROOT.mkdir(parents=True, exist_ok=True)

ff1.Cache.enable_cache(str(CACHE_DIR))

cache_info = ff1.Cache.get_cache_info()

logger.info("FastF1 cache enabled: %s @ %s", cache_info[0], cache_info[1])

# --- CONFIG ---
YEARS = range(2025, 2026)
SESSION_TYPES = ["R", "Q", "FP1", "FP2", "FP3", "Sprint"]  # Sprint only 2021+
SESSION_DATASETS = ("RESULTS", "LAPS", "TELEMETRY", "WEATHER")
MASTER_DATASETS = ("EVENT_SCHEDULE", "DRIVER_INFO", "CIRCUIT_INFO")
RATE_LIMIT_WAIT_SECONDS = 60

# --- HELPER FUNCTIONS ---
def to_dataframe(obj: Any) -> Optional[pd.DataFrame]:
    """Convert object to DataFrame if possible."""
    if obj is None:
        return None
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    else:
        try:
            df = obj.copy()
            if not isinstance(df, pd.DataFrame):
                df = pd.DataFrame(df)
        except Exception:
            try:
                df = pd.DataFrame(obj)
            except Exception:
                return None
    if hasattr(df, "empty") and df.empty:
        return None
    return df.reset_index(drop=True)


def handle_rate_limit(func, description: str, *args, **kwargs) -> Any:
    """Simple rate limit handler - only retries if 'limit' is in error message."""
    while True:
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            exc_msg = str(exc).lower()
            if "limit" in exc_msg:
                logger.warning(
                    "%s → rate limit detected; waiting %ds before retry",
                    description,
                    RATE_LIMIT_WAIT_SECONDS,
                )
                time.sleep(RATE_LIMIT_WAIT_SECONDS)
                continue
            # For other errors, skip the data as it may be missing
            logger.info("%s → error (not rate limit): %s; skipping", description, exc)
            return None


# --- DATA COLLECTOR CLASSES ---
class SessionDataCollector:
    """Collects session-specific data: Results, Laps, Telemetry, Weather."""
    
    def __init__(self, save_root: Path):
        self.save_root = save_root
        self.data_store: Dict[str, List[pd.DataFrame]] = {
            key: [] for key in SESSION_DATASETS
        }
    
    def needs_dataset(self, dataset: str, year: int) -> bool:
        """Check if dataset file doesn't exist yet."""
        path = self.save_root / f"ALL_{dataset}_{year}.csv"
        return not path.exists()
    
    def get_required_datasets(self, year: int) -> Set[str]:
        """Get list of datasets that need to be collected."""
        return {
            ds for ds in SESSION_DATASETS 
            if self.needs_dataset(ds, year)
        }
    
    def collect_session_data(
        self, 
        session: Session, 
        year: int, 
        event_name: str, 
        session_type: str
    ) -> None:
        """Collect all available data from a session."""
        session_label = f"{year} {event_name} {session_type}"
        required = self.get_required_datasets(year)
        
        if not required:
            return
        
        # Determine what to load
        load_laps = bool({"LAPS", "TELEMETRY", "WEATHER"} & required)
        load_telemetry = "TELEMETRY" in required
        load_weather = "WEATHER" in required
        load_results = "RESULTS" in required
        
        if not any([load_laps, load_telemetry, load_weather, load_results]):
            return
        
        # Load session data
        def load_func():
            session.load(
                laps=load_laps,
                telemetry=load_telemetry,
                weather=load_weather,
                messages=False  # Not collecting messages
            )
            return True
        
        result = handle_rate_limit(load_func, f"{session_label} load()")
        if result is None:
            logger.info("    %s → load() failed; skipping", session_label)
            return
        
        # Collect Results
        if "RESULTS" in required:
            self._collect_results(session, year, event_name, session_type, session_label)
        
        # Collect Laps
        if "LAPS" in required:
            self._collect_laps(session, year, event_name, session_type, session_label)
        
        # Collect Telemetry
        if "TELEMETRY" in required:
            self._collect_telemetry(session, year, event_name, session_type, session_label)
        
        # Collect Weather
        if "WEATHER" in required:
            self._collect_weather(session, year, event_name, session_type, session_label)
    
    def _collect_results(
        self, session: Session, year: int, event_name: str, 
        session_type: str, session_label: str
    ) -> None:
        """Collect session results."""
        try:
            results = session.results
            df = to_dataframe(results)
            if df is not None:
                df["Year"] = year
                df["Event"] = event_name
                df["Session"] = session_type
                self.data_store["RESULTS"].append(df)
                logger.info("    %s → RESULTS: %d rows", session_label, len(df))
        except Exception as exc:
            logger.info("    %s → RESULTS: no data (%s)", session_label, exc)
    
    def _collect_laps(
        self, session: Session, year: int, event_name: str,
        session_type: str, session_label: str
    ) -> None:
        """Collect lap data."""
        try:
            laps = session.laps
            df = to_dataframe(laps)
            if df is not None:
                df["Year"] = year
                df["Event"] = event_name
                df["Session"] = session_type
                self.data_store["LAPS"].append(df)
                logger.info("    %s → LAPS: %d rows", session_label, len(df))
        except Exception as exc:
            logger.info("    %s → LAPS: no data (%s)", session_label, exc)
    
    def _collect_telemetry(
        self, session: Session, year: int, event_name: str,
        session_type: str, session_label: str
    ) -> None:
        """Collect telemetry data."""
        try:
            car_data = session.car_data
            if isinstance(car_data, dict) and car_data:
                for driver, tel in car_data.items():
                    df = to_dataframe(tel)
                    if df is not None:
                        df["Year"] = year
                        df["Event"] = event_name
                        df["Session"] = session_type
                        df["Driver"] = driver
                        self.data_store["TELEMETRY"].append(df)
                        logger.info("    %s → TELEMETRY [%s]: %d rows", session_label, driver, len(df))
            else:
                df = to_dataframe(car_data)
                if df is not None:
                    df["Year"] = year
                    df["Event"] = event_name
                    df["Session"] = session_type
                    self.data_store["TELEMETRY"].append(df)
                    logger.info("    %s → TELEMETRY: %d rows", session_label, len(df))
        except Exception as exc:
            logger.info("    %s → TELEMETRY: no data (%s)", session_label, exc)
    
    def _collect_weather(
        self, session: Session, year: int, event_name: str,
        session_type: str, session_label: str
    ) -> None:
        """Collect weather data."""
        try:
            weather = session.weather_data
            df = to_dataframe(weather)
            if df is not None:
                df["Year"] = year
                df["Event"] = event_name
                df["Session"] = session_type
                self.data_store["WEATHER"].append(df)
                logger.info("    %s → WEATHER: %d rows", session_label, len(df))
        except Exception as exc:
            logger.info("    %s → WEATHER: no data (%s)", session_label, exc)
    
    def save_year_data(self, year: int) -> None:
        """Save all collected data for a year to CSV files."""
        # Only save datasets that were actually being collected
        required = self.get_required_datasets(year)
        
        for dataset, frames in self.data_store.items():
            if dataset not in required:
                continue  # Skip datasets we weren't collecting
            if not frames:
                logger.warning("  [NO DATA] %s for %s", dataset, year)
                continue
            # ... rest of save logic
            
            combined = pd.concat(frames, ignore_index=True, sort=False)
            path = self.save_root / f"ALL_{dataset}_{year}.csv"
            combined.to_csv(path, index=False)
            logger.info("  [SAVED] %s: %d rows → %s", dataset, len(combined), path.name)
        
        # Clear data store for next year
        self.data_store = {key: [] for key in SESSION_DATASETS}


class EventScheduleCollector:
    """Collects event schedule data across all years."""
    
    def __init__(self, save_root: Path):
        self.save_root = save_root
        self.schedules: List[pd.DataFrame] = []
    
    def collect_year_schedule(self, year: int) -> None:
        """Collect event schedule for a year."""
        def get_schedule():
            return ff1.get_event_schedule(year, include_testing=False)
        
        schedule = handle_rate_limit(get_schedule, f"{year} schedule")
        if schedule is not None:
            df = to_dataframe(schedule)
            if df is not None:
                df["Year"] = year
                self.schedules.append(df)
                logger.info("  → Event Schedule %s: %d events", year, len(df))
    
    def save_all(self) -> None:
        """Save combined event schedule to CSV."""
        if not self.schedules:
            logger.warning("  [NO DATA] EVENT_SCHEDULE")
            return
        
        combined = pd.concat(self.schedules, ignore_index=True, sort=False)
        path = self.save_root / "ALL_EVENT_SCHEDULE.csv"

        # IMPORTANT: don't wipe prior seasons if this run only extracted a subset of years.
        if path.exists():
            try:
                existing = pd.read_csv(path, low_memory=False)
                merged = pd.concat([existing, combined], ignore_index=True, sort=False)

                # Prefer the newest scrape for the same season+event+round keys.
                key_cols = [c for c in ["Year", "EventName", "RoundNumber"] if c in merged.columns]
                if key_cols:
                    merged = merged.sort_values(key_cols).drop_duplicates(subset=key_cols, keep="last")
                else:
                    merged = merged.drop_duplicates()

                combined = merged
            except Exception as exc:
                logger.warning("  [WARN] Could not merge existing %s (%s); overwriting", path.name, exc)

        combined.to_csv(path, index=False)
        logger.info("  [SAVED] EVENT_SCHEDULE: %d rows → %s", len(combined), path.name)


class DriverInfoCollector:
    """Collects driver information across all sessions."""
    
    def __init__(self, save_root: Path):
        self.save_root = save_root
        self.driver_info_list: List[Dict[str, Any]] = []
        self.seen_drivers: Set[str] = set()
    
    def collect_driver_info(
        self, session: Session, year: int, event_name: str, session_type: str
    ) -> None:
        """Collect driver info from session results."""
        try:
            results = session.results
            if results is None:
                return
            
            for _, row in results.iterrows():
                driver_id = row.get("Abbreviation") or row.get("DriverNumber")
                if not driver_id or str(driver_id) in self.seen_drivers:
                    continue
                
                def get_driver():
                    return session.get_driver(driver_id)
                
                driver_info = handle_rate_limit(get_driver, f"Driver {driver_id}")
                if driver_info is not None:
                    # Convert driver info to dict/Series
                    if hasattr(driver_info, "to_dict"):
                        info_dict = driver_info.to_dict()
                    elif isinstance(driver_info, dict):
                        info_dict = driver_info
                    else:
                        info_dict = {"DriverInfo": str(driver_info)}
                    
                    info_dict["Year"] = year
                    info_dict["Event"] = event_name
                    info_dict["Session"] = session_type
                    info_dict["DriverId"] = driver_id
                    self.driver_info_list.append(info_dict)
                    self.seen_drivers.add(str(driver_id))
        except Exception as exc:
            logger.debug("Driver info collection error: %s", exc)
    
    def save_all(self) -> None:
        """Save combined driver info to CSV."""
        if not self.driver_info_list:
            logger.warning("  [NO DATA] DRIVER_INFO")
            return
        
        df = pd.DataFrame(self.driver_info_list)
        path = self.save_root / "ALL_DRIVER_INFO.csv"
        df.to_csv(path, index=False)
        logger.info("  [SAVED] DRIVER_INFO: %d rows → %s", len(df), path.name)


class CircuitInfoCollector:
    """Collects circuit information for each event."""
    
    def __init__(self, save_root: Path):
        self.save_root = save_root
        self.circuit_info_list: List[Dict[str, Any]] = []
        self.seen_circuits: Set[tuple] = set()  # (year, event_name)
    
    def collect_circuit_info(
        self, session: Session, year: int, event_name: str, session_type: str
    ) -> None:
        """Collect circuit info from session."""
        circuit_key = (year, event_name)
        if circuit_key in self.seen_circuits:
            return
    
    # def collect_circuit_info(
    #     self, session: Session, year: int, event_name: str, session_type: str
    # ) -> None:
    #     """Collect circuit info from session."""
    #     circuit_key = (year, event_name)
    #     if circuit_key in self.seen_circuits:
    #         return

    #     # MUST load session before get_circuit_info
    #     session.load(laps=True, telemetry=False, weather=False, messages=False)

    #     circuit_info = session.get_circuit_info()

    #     # keep original dict, but add circuit name too
    #     if hasattr(circuit_info, "to_dict"):
    #         info_dict = circuit_info.to_dict()
    #     elif isinstance(circuit_info, dict):
    #         info_dict = circuit_info
    #     else:
    #         info_dict = {"CircuitInfo": str(circuit_info)}

    #     add name/location/country explicitly
    #     info_dict["CircuitName"] = getattr(circuit_info, "Name", None)
    #     info_dict["Location"] = getattr(circuit_info, "Location", None)
    #     info_dict["Country"] = getattr(circuit_info, "Country", None)
    #     # ev = session.event
    #     # info_dict["CircuitName"] = ev.get("CircuitName") or ev.get("EventName")
    #     # info_dict["Location"] = ev.get("Location")
    #     # info_dict["Country"] = ev.get("Country")
        

    #     info_dict["Year"] = year
    #     info_dict["Event"] = event_name
    #     info_dict["Session"] = session_type
    #     self.circuit_info_list.append(info_dict)
    #     self.seen_circuits.add(circuit_key)
        
        def get_circuit():
            return session.get_circuit_info()
        
        circuit_info = handle_rate_limit(get_circuit, f"Circuit {year} {event_name}")
        if circuit_info is not None:
            # Convert circuit info to dict
            if hasattr(circuit_info, "to_dict"):
                info_dict = circuit_info.to_dict()
            elif isinstance(circuit_info, dict):
                info_dict = circuit_info
            else:
                info_dict = {"CircuitInfo": str(circuit_info)}
            
            info_dict["Year"] = year
            info_dict["Event"] = event_name
            info_dict["Session"] = session_type
            self.circuit_info_list.append(info_dict)
            self.seen_circuits.add(circuit_key)
    
    def save_all(self) -> None:
        """Save combined circuit info to CSV."""
        if not self.circuit_info_list:
            logger.warning("  [NO DATA] CIRCUIT_INFO")
            return
        
        df = pd.DataFrame(self.circuit_info_list)
        path = self.save_root / "ALL_CIRCUIT_INFO.csv"
        df.to_csv(path, index=False)
        logger.info("  [SAVED] CIRCUIT_INFO: %d rows → %s", len(df), path.name)


# --- MAIN LOOP ---
# Initialize collectors
session_collector = SessionDataCollector(SAVE_ROOT)
schedule_collector = EventScheduleCollector(SAVE_ROOT)
driver_collector = DriverInfoCollector(SAVE_ROOT)
circuit_collector = CircuitInfoCollector(SAVE_ROOT)

# Process each year
for year in YEARS:
    logger.info("\n%s YEAR %s %s", "=" * 20, year, "=" * 20)
    year_start = time.perf_counter()
    
    # Check if we need to collect any session data
    required_datasets = session_collector.get_required_datasets(year)
    
    if not required_datasets:
        logger.info("  All session dataset CSVs already exist for %s; skipping year", year)
        # Still collect schedule, driver, and circuit info if needed
        schedule_collector.collect_year_schedule(year)
        continue
    
    # if not required_datasets:
    #     logger.info("  All session dataset CSVs already exist for %s; skipping year", year)
    #     # Still collect schedule, driver, and circuit info
    #     schedule_collector.collect_year_schedule(year)

    #     # Use the already-fetched schedule to collect circuit info
    #     schedule_df = schedule_collector.schedules[-1]
    #     for _, ev in schedule_df.iterrows():
    #         ev_name = ev["EventName"]
    #         session = handle_rate_limit(lambda: ff1.get_session(year, ev_name, "R"),
    #                                     f"{year} {ev_name} get_session()")
    #         if session is None:
    #             continue

    #         # Force minimal load for circuit info
    #         session.load(laps=True, telemetry=False, weather=False, messages=False)

    #         # Collect circuit info only
    #         circuit_collector.collect_circuit_info(session, year, ev_name, "R")

    #     continue
    
    logger.info("  Pending datasets: %s", ", ".join(sorted(required_datasets)))
    
    # Get event schedule for this year
    schedule_collector.collect_year_schedule(year)
    
    # Get schedule DataFrame for iteration
    def get_schedule():
        return ff1.get_event_schedule(year, include_testing=False)
    
    schedule = handle_rate_limit(get_schedule, f"{year} schedule")
    if schedule is None:
        logger.warning("  Failed to get schedule for %s; skipping year", year)
        continue
    
    schedule_df = to_dataframe(schedule)
    if schedule_df is None:
        logger.warning("  Schedule is empty for %s; skipping year", year)
        continue
    
    processed_sessions = 0
    
    # Process each event
    for _, ev in schedule_df.iterrows():
        ev_name = ev["EventName"]
        logger.info("  → %s", ev_name)
        
        # Process each session type
        for sess_type in SESSION_TYPES:
            if sess_type == "Sprint" and year < 2021:
                continue
            
            session_label = f"{year} {ev_name} {sess_type}"
            
            # Get session
            def get_session():
                return ff1.get_session(year, ev_name, sess_type)
            
            session = handle_rate_limit(get_session, f"{session_label} get_session()")
            if session is None:
                logger.info("    %s → session unavailable; skipping", session_label)
                continue
            
            # Collect session data
            session_collector.collect_session_data(session, year, ev_name, sess_type)
            
            # Collect driver info (only once per driver)
            driver_collector.collect_driver_info(session, year, ev_name, sess_type)
            
            # Collect circuit info (only once per event)
            circuit_collector.collect_circuit_info(session, year, ev_name, sess_type)
            
            processed_sessions += 1
    
    # Save session data for this year
    session_collector.save_year_data(year)
    
    year_elapsed = time.perf_counter() - year_start
    logger.info(
        "Completed %s with %d sessions in %.1fs",
        year,
        processed_sessions,
        year_elapsed,
    )

# Save master datasets (across all years)
logger.info("\n%s SAVING MASTER DATASETS %s", "=" * 20, "=" * 20)
schedule_collector.save_all()
driver_collector.save_all()
circuit_collector.save_all()

logger.info("\n=== ALL DONE ===")



00:07:35 |     INFO | FastF1 cache enabled: C:\Users\erikv\Downloads\F1\notebooks\f1_cache @ 55218878900
00:07:35 |     INFO | 
==================== YEAR 2025 ====================
00:07:35 |     INFO |   All session dataset CSVs already exist for 2025; skipping year
00:07:36 |     INFO |   → Event Schedule 2025: 24 events
core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.1]
00:07:36 |     INFO | Loading data for Australian Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
00:07:36 |     INFO | Using cached data for session_info
req            INFO 	Using cached data for driver_info
00:07:36 |     INFO | Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
00:07:37 |     INFO | Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
00:07:37 |     INFO | Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
00:0

: 